# MediTOD -- Dialogue Act Classification (DAC) Training

> **Model:** `PubMedBERT-base-uncased` (Sequence Classification)  
> **Task:** 16-class Dialogue Act Classification on Indian English medical dialogues  
> **Dataset:** 204 synthetic dialogues (5,070 utterances) -- Claude Sonnet 4.6  
> **Platform:** Kaggle T4 GPU

---

### Pipeline
```
Input Utterance (with 2-turn context)
    |
PubMedBERT Tokenizer (max_len=128)
    |
PubMedBERT Encoder (bottom 6 frozen)
    |
[CLS] -> Dropout -> Linear(768 -> 16)
    |
Focal Loss + sqrt-softened class weights
    |
Predicted Dialogue Act (1 of 16 classes)
```

---
## Cell 1 -- Install Dependencies

In [ ]:
!pip install -q transformers accelerate scikit-learn seaborn matplotlib

---
## Cell 2 -- Imports & Configuration

In [ ]:
# ==============================================================
#  IMPORTS
# ==============================================================
import os
import json
import random
import logging
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ==============================================================
#  LOGGING
# ==============================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# ==============================================================
#  REPRODUCIBILITY
# ==============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ==============================================================
#  DEVICE
# ==============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# ==============================================================
#  HYPERPARAMETERS
# ==============================================================
MODEL_NAME      = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
MAX_SEQ_LEN     = 192       # v2: increased for 4 turns (3 context + 1 target)
CONTEXT_WINDOW  = 3         # v2: increased from 2 for better narrative context
BATCH_SIZE      = 32
NUM_EPOCHS      = 15
BACKBONE_LR     = 1e-5      # lower LR for pretrained layers
CLASSIFIER_LR   = 5e-5      # higher LR for classification head
WEIGHT_DECAY    = 0.05      # v2.1: slightly more weight decay to generalize better
WARMUP_RATIO    = 0.10
FOCAL_GAMMA     = 3.0       # v2: increased from 2.0 to focus harder on rare classes
DROPOUT         = 0.2       # v2.1: higher dropout on classifier to prevent overfitting
PATIENCE        = 5         # early stopping patience
FREEZE_LAYERS   = 4         # v2: reduced from 6 — more flexibility for subtle intent boundaries

# ==============================================================
#  16 DIALOGUE ACT TAGS
# ==============================================================
TAGS = [
    'Request-Etiology', 'Request-Precautions', 'Request-Medical_Advice',
    'Inform-Etiology', 'Diagnose', 'Request-Basic_Information',
    'Request-Drug_Recommendation', 'Inform-Medical_Advice',
    'Request-Existing_Examination_and_Treatment', 'Inform-Basic_Information',
    'Inform-Precautions', 'Inform-Existing_Examination_and_Treatment',
    'Inform-Drug_Recommendation', 'Request-Symptom', 'Inform-Symptom', 'Other'
]

tag2id = {tag: idx for idx, tag in enumerate(TAGS)}
id2tag = {idx: tag for tag, idx in tag2id.items()}
NUM_LABELS = len(TAGS)

logger.info(f"Model: {MODEL_NAME}")
logger.info(f"Labels: {NUM_LABELS} classes")
logger.info(f"Max seq len: {MAX_SEQ_LEN}, Context window: {CONTEXT_WINDOW}")

# -- Kaggle Dataset Path --
DATA_DIR = '/kaggle/input/meditod-dac-dataset'

---
## Cell 3 -- Data Loading & Context Window Construction

In [ ]:
# ==============================================================
#  DATA LOADING & CONTEXT WINDOW CONSTRUCTION
# ==============================================================

def load_jsonl(path):
    """Load a JSONL file into a list of dicts."""
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def build_context_examples(dialogues, context_window=CONTEXT_WINDOW):
    """
    v2: Added speaker-role tokens to each turn for better intent disambiguation.
    
    Input format:
        [DOCTOR] prev question [SEP] [PATIENT] prev answer [SEP] [DOCTOR] target turn
    
    Returns: list of dict with keys 'text' and 'label'
    """
    examples = []
    for d in dialogues:
        utts = d['utterances']
        for i, u in enumerate(utts):
            parts = []
            start = max(0, i - context_window)
            for j in range(start, i):
                spk = utts[j].get('speaker', 'Unknown').upper()
                spk_token = f"[{spk}]"
                parts.append(f"{spk_token} {utts[j]['text']}")
            # Target turn
            spk = u.get('speaker', 'Unknown').upper()
            spk_token = f"[{spk}]"
            parts.append(f"{spk_token} {u['text']}")
            full_text = " [SEP] ".join(parts)
            label = tag2id[u['intent']]
            examples.append({'text': full_text, 'label': label})
    return examples

# -- Load raw dialogues --
train_dialogues = load_jsonl(os.path.join(DATA_DIR, 'train.jsonl'))
val_dialogues   = load_jsonl(os.path.join(DATA_DIR, 'val.jsonl'))
test_dialogues  = load_jsonl(os.path.join(DATA_DIR, 'test.jsonl'))

logger.info(f"Loaded: {len(train_dialogues)} train, "
            f"{len(val_dialogues)} val, {len(test_dialogues)} test dialogues")

# -- Build context-enriched examples --
train_examples = build_context_examples(train_dialogues)
val_examples   = build_context_examples(val_dialogues)
test_examples  = build_context_examples(test_dialogues)

logger.info(f"Training examples:   {len(train_examples)}")
logger.info(f"Validation examples: {len(val_examples)}")
logger.info(f"Test examples:       {len(test_examples)}")

# -- Class distribution --
print(f"\n{'=' * 70}")
print(f"  TRAINING SET CLASS DISTRIBUTION")
print(f"{'=' * 70}")
train_label_counts = Counter(ex['label'] for ex in train_examples)
for tag_id in sorted(train_label_counts.keys()):
    tag_name = id2tag[tag_id]
    count = train_label_counts[tag_id]
    pct = 100 * count / len(train_examples)
    bar = '#' * int(pct * 2)
    print(f"  {tag_name:45s} {count:5d}  ({pct:5.1f}%)  {bar}")

# -- Sample inputs --
print(f"\n{'=' * 70}")
print(f"  SAMPLE CONTEXT-ENRICHED INPUTS")
print(f"{'=' * 70}")
for i in range(min(3, len(train_examples))):
    ex = train_examples[i]
    print(f"\n  Example {i+1}:")
    print(f"  Label: {id2tag[ex['label']]}")
    text_preview = ex['text'][:180] + ('...' if len(ex['text']) > 180 else '')
    print(f"  Input: {text_preview}")

---
## Cell 4 -- Tokenization & Dataset

In [ ]:
# ==============================================================
#  TOKENIZER & PYTORCH DATASET
# ==============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class DACDataset(Dataset):
    """PyTorch dataset for Dialogue Act Classification."""
    def __init__(self, examples, tokenizer, max_len=MAX_SEQ_LEN):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        ex = self.examples[idx]
        encoding = self.tokenizer(
            ex['text'],
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels':         torch.tensor(ex['label'], dtype=torch.long)
        }

# -- Create datasets & dataloaders --
train_dataset = DACDataset(train_examples, tokenizer)
val_dataset   = DACDataset(val_examples, tokenizer)
test_dataset  = DACDataset(test_examples, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,   shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

logger.info(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

# -- Verify --
sample = train_dataset[0]
print(f"Sample input_ids shape:      {sample['input_ids'].shape}")
print(f"Sample attention_mask shape: {sample['attention_mask'].shape}")
print(f"Sample label: {sample['labels'].item()} ({id2tag[sample['labels'].item()]})")

# -- Tokenization length analysis --
token_lengths = []
for ex in train_examples[:500]:
    enc = tokenizer(ex['text'], truncation=False)
    token_lengths.append(len(enc['input_ids']))

print(f"\nTokenization length statistics (first 500 train examples):")
print(f"  Mean:  {np.mean(token_lengths):.1f}")
print(f"  Median: {np.median(token_lengths):.0f}")
print(f"  95th pctile: {np.percentile(token_lengths, 95):.0f}")
print(f"  99th pctile: {np.percentile(token_lengths, 99):.0f}")
print(f"  Max:   {max(token_lengths)}")
print(f"  Truncated (>{MAX_SEQ_LEN}): {sum(1 for l in token_lengths if l > MAX_SEQ_LEN)} / {len(token_lengths)}")

---
## Cell 5 -- Model, Loss Function & Optimizer

In [ ]:
# ==============================================================
#  FOCAL LOSS
# ==============================================================
class FocalLoss(nn.Module):
    """Focal Loss with per-class weighting for class imbalance."""
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
    
    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(
            logits, targets,
            weight=self.alpha,
            reduction='none'
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# ==============================================================
#  MODEL -- BERT + [CLS] Classifier
# ==============================================================
class BertDAC(nn.Module):
    """PubMedBERT-based Dialogue Act Classifier using [CLS] pooling."""
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output  # [CLS] representation
        return self.classifier(self.dropout(pooled))

model = BertDAC(MODEL_NAME, NUM_LABELS, dropout=DROPOUT).to(device)

# ==============================================================
#  LAYER FREEZING -- Freeze embeddings + bottom 6 layers
# ==============================================================
for name, param in model.named_parameters():
    if "bert.embeddings" in name:
        param.requires_grad = False
    elif "bert.encoder.layer" in name:
        layer_num = int(name.split("bert.encoder.layer.")[1].split(".")[0])
        if layer_num < FREEZE_LAYERS:
            param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
logger.info(f"Trainable: {trainable:,} / {total_params:,} ({100*trainable/total_params:.1f}%)")

# ==============================================================
#  CLASS WEIGHTS & LOSS
# ==============================================================
train_labels_array = np.array([ex['label'] for ex in train_examples])
raw_weights = compute_class_weight(
    'balanced', classes=np.arange(NUM_LABELS), y=train_labels_array
)
soft_weights = np.sqrt(raw_weights)  # sqrt-softening prevents overcorrection
class_weights = torch.tensor(soft_weights, dtype=torch.float).to(device)

print(f"\n{'=' * 70}")
print(f"  CLASS WEIGHTS (sqrt-softened)")
print(f"{'=' * 70}")
for i, w in enumerate(soft_weights):
    print(f"  {TAGS[i]:45s}  weight = {w:.3f}")

criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=class_weights)

# ==============================================================
#  OPTIMIZER & SCHEDULER (Differential LR + Cosine Warmup)
# ==============================================================
optimizer_groups = [
    {
        "params": [p for n, p in model.named_parameters()
                   if p.requires_grad and "classifier" not in n],
        "lr": BACKBONE_LR,
        "weight_decay": WEIGHT_DECAY,
    },
    {
        "params": [p for n, p in model.named_parameters()
                   if p.requires_grad and "classifier" in n],
        "lr": CLASSIFIER_LR,
        "weight_decay": WEIGHT_DECAY,
    },
]
optimizer = torch.optim.AdamW(optimizer_groups)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

logger.info(f"Total steps: {total_steps}, Warmup steps: {warmup_steps}")
logger.info(f"Backbone LR: {BACKBONE_LR}, Classifier LR: {CLASSIFIER_LR}")

---
## Cell 6 -- Training Loop with Detailed Logging

In [ ]:
# ==============================================================
#  EVALUATION HELPER
# ==============================================================
def evaluate(model, loader, criterion):
    """Run evaluation and return loss, F1, accuracy, preds, labels."""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
    
    avg_loss = total_loss / len(loader)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted', zero_division=0
    )
    acc = accuracy_score(all_labels, all_preds)
    
    return avg_loss, f1, acc, all_preds, all_labels

# ==============================================================
#  TRAINING LOOP
# ==============================================================
history = {
    'train_loss': [], 'val_loss': [],
    'val_f1': [], 'val_acc': []
}

best_val_f1 = 0.0
best_epoch = 0
patience_counter = 0

logger.info(f"\n{'=' * 70}")
logger.info(f"  STARTING TRAINING -- {NUM_EPOCHS} epochs, patience={PATIENCE}")
logger.info(f"{'=' * 70}\n")

scaler = torch.amp.GradScaler('cuda')  # FP16 mixed precision

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0
    num_batches = 0
    epoch_preds = []
    epoch_labels = []
    
    for batch_idx, batch in enumerate(train_loader, 1):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        epoch_loss += loss.item()
        num_batches += 1
        
        preds = torch.argmax(logits, dim=-1)
        epoch_preds.extend(preds.cpu().numpy().tolist())
        epoch_labels.extend(labels.cpu().numpy().tolist())
        
        # Log every 25 batches
        if batch_idx % 25 == 0 or batch_idx == len(train_loader):
            current_lr = scheduler.get_last_lr()[0]
            batch_acc = accuracy_score(
                labels.cpu().numpy(), preds.cpu().numpy()
            )
            print(f"  Epoch {epoch:2d} | Batch {batch_idx:3d}/{len(train_loader)} | "
                  f"Loss: {loss.item():.4f} | Acc: {batch_acc:.3f} | LR: {current_lr:.2e}")
    
    avg_train_loss = epoch_loss / num_batches
    train_acc = accuracy_score(epoch_labels, epoch_preds)
    
    # -- Validation --
    val_loss, val_f1, val_acc, _, _ = evaluate(model, val_loader, criterion)
    
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    history['val_acc'].append(val_acc)
    
    # -- Epoch summary --
    is_best = val_f1 > best_val_f1
    print(f"\n{'-' * 70}")
    print(f"  Epoch {epoch:2d}/{NUM_EPOCHS} Summary")
    print(f"{'-' * 70}")
    print(f"  Train Loss:  {avg_train_loss:.4f}   |   Train Acc:  {train_acc:.4f}")
    print(f"  Val Loss:    {val_loss:.4f}   |   Val Acc:    {val_acc:.4f}")
    print(f"  Val F1:      {val_f1:.4f}   {'    << NEW BEST!' if is_best else ''}")
    print(f"{'-' * 70}\n")
    
    # -- Early stopping --
    if is_best:
        best_val_f1 = val_f1
        best_epoch = epoch
        patience_counter = 0
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
            'val_acc': val_acc,
            'val_loss': val_loss,
        }, 'best_dac_model.pt')
        logger.info(f"  >> Best model saved (epoch {epoch}, F1={val_f1:.4f})")
    else:
        patience_counter += 1
        logger.info(f"  No improvement. Patience: {patience_counter}/{PATIENCE}")
        
        if patience_counter >= PATIENCE:
            logger.info(f"\n  >> Early stopping triggered at epoch {epoch}")
            break

print(f"\n{'=' * 70}")
print(f"  TRAINING COMPLETE")
print(f"  Best epoch: {best_epoch}, Best Val F1: {best_val_f1:.4f}")
print(f"{'=' * 70}")

---
## Cell 7 -- Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss curves
axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', color='#FF6B6B', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss',   color='#4ECDC4', linewidth=2)
axes[0].axvline(x=best_epoch, color='gold', linestyle='--', label=f'Best (epoch {best_epoch})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1 curve
axes[1].plot(epochs_range, history['val_f1'], label='Val Weighted F1', color='#45B7D1', linewidth=2, marker='o')
axes[1].axvline(x=best_epoch, color='gold', linestyle='--', label=f'Best (epoch {best_epoch})')
axes[1].axhline(y=best_val_f1, color='gold', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Weighted F1')
axes[1].set_title(f'Validation F1 (best: {best_val_f1:.4f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Accuracy curve
axes[2].plot(epochs_range, history['val_acc'], label='Val Accuracy', color='#96CEB4', linewidth=2, marker='s')
axes[2].axvline(x=best_epoch, color='gold', linestyle='--', label=f'Best (epoch {best_epoch})')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Validation Accuracy')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('DAC Training Metrics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Training curves saved to training_curves.png")

---
## Cell 8 -- Evaluation on Test Set

In [ ]:
# ==============================================================
#  LOAD BEST CHECKPOINT & EVALUATE ON TEST SET
# ==============================================================
checkpoint = torch.load('best_dac_model.pt', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']} (Val F1: {checkpoint['val_f1']:.4f})")

test_loss, test_f1, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion)

print(f"\n{'=' * 70}")
print(f"  TEST SET RESULTS")
print(f"{'=' * 70}")
print(f"  Test Loss:      {test_loss:.4f}")
print(f"  Test F1:        {test_f1:.4f}")
print(f"  Test Accuracy:  {test_acc:.4f}")
print(f"{'=' * 70}")

# -- Detailed per-class classification report --
print(f"\n--- Detailed Classification Report ---\n")
report_text = classification_report(
    test_labels, test_preds,
    target_names=TAGS, digits=4, zero_division=0
)
print(report_text)

# Also compute macro F1
_, _, macro_f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average='macro', zero_division=0
)
print(f"Macro F1: {macro_f1:.4f}")

---
## Cell 9 -- Confusion Matrix

In [ ]:
# ==============================================================
#  CONFUSION MATRIX
# ==============================================================
cm = confusion_matrix(test_labels, test_preds, labels=list(range(NUM_LABELS)))

# Shorten tag names for readability
short_names = [t.replace('Request-', 'Req-').replace('Inform-', 'Inf-')
               .replace('Existing_Examination_and_Treatment', 'ExamTreat')
               .replace('Drug_Recommendation', 'DrugRec')
               .replace('Medical_Advice', 'MedAdv')
               .replace('Basic_Information', 'BasicInfo')
               for t in TAGS]

plt.figure(figsize=(16, 13))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=short_names, yticklabels=short_names,
    linewidths=0.5, cbar_kws={'shrink': 0.7}
)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.title(f'DAC Confusion Matrix (Test Set, F1={test_f1:.4f})', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved to confusion_matrix.png")

# -- Per-class accuracy --
print(f"\n{'=' * 60}")
print(f"  PER-CLASS ACCURACY")
print(f"{'=' * 60}")
for i in range(NUM_LABELS):
    class_total = cm[i].sum()
    class_correct = cm[i][i]
    class_acc = class_correct / class_total if class_total > 0 else 0
    print(f"  {TAGS[i]:45s}  {class_correct:3d}/{class_total:3d}  ({class_acc:.1%})")

---
## Cell 10 -- Inference Examples

In [ ]:
# ==============================================================
#  INFERENCE ON EXAMPLE MEDICAL DIALOGUE TURNS
# ==============================================================
def predict_turn(text, context_turns=None):
    """Predict dialogue act for a single utterance with optional context.
    
    Args:
        text: The target utterance (e.g., "Patient: I have a fever")
        context_turns: Optional list of preceding turns for context
    
    Returns:
        (predicted_tag, confidence, top3_predictions)
    """
    model.eval()
    parts = []
    if context_turns:
        for turn in context_turns[-CONTEXT_WINDOW:]:
            parts.append(turn)
    parts.append(text)
    full_input = " [SEP] ".join(parts)
    
    encoding = tokenizer(
        full_input, truncation=True, max_length=MAX_SEQ_LEN,
        padding='max_length', return_tensors='pt'
    )
    
    with torch.no_grad():
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)
        logits = model(input_ids, attention_mask)
        probs = F.softmax(logits, dim=-1)[0]
        
        top3_vals, top3_ids = torch.topk(probs, 3)
        top3 = [(id2tag[idx.item()], val.item()) for idx, val in zip(top3_ids, top3_vals)]
    
    pred_id = top3_ids[0].item()
    return id2tag[pred_id], top3[0][1], top3

# -- Test examples --
test_cases = [
    {
        "context": ["Doctor: What brings you here today?"],
        "target": "Patient: Doctor, I have been having severe headache and fever for three days now.",
    },
    {
        "context": [
            "Patient: I have chest pain and breathlessness.",
            "Doctor: How long has this been going on?",
        ],
        "target": "Patient: For about two weeks now, it gets worse at night.",
    },
    {
        "context": ["Patient: The cough is not going away."],
        "target": "Doctor: I think you might have bronchitis. Let me prescribe some antibiotics.",
    },
    {
        "context": ["Doctor: Do you smoke or drink?"],
        "target": "Patient: I used to smoke about 10 cigarettes daily but I stopped last year.",
    },
    {
        "context": ["Patient: What should I do to prevent this from happening again?"],
        "target": "Doctor: Avoid spicy and oily food, drink plenty of water, and maintain regular meal timings.",
    },
    {
        "context": [],
        "target": "Doctor: Have you ever been diagnosed with diabetes or hypertension?",
    },
    {
        "context": ["Doctor: Any allergies to medications?"],
        "target": "Patient: Yes, I had a reaction to penicillin once.",
    },
]

print(f"{'=' * 70}")
print(f"  INFERENCE EXAMPLES")
print(f"{'=' * 70}")

for i, tc in enumerate(test_cases, 1):
    pred_tag, confidence, top3 = predict_turn(tc["target"], tc.get("context"))
    
    print(f"\n  --- Example {i} ---")
    if tc.get('context'):
        for ctx in tc['context']:
            print(f"  Context:    {ctx}")
    print(f"  Target:     {tc['target']}")
    print(f"  Predicted:  {pred_tag} ({confidence:.1%})")
    top3_str = ', '.join(f'{t} ({c:.1%})' for t, c in top3)
    print(f"  Top-3:      {top3_str}")

---
## Cell 11 -- Save Best Weights & Reports

In [ ]:
# ==============================================================
#  SAVE EVERYTHING FOR DOWNLOAD
# ==============================================================
import shutil

SAVE_DIR = '/kaggle/working/meditod_dac_output'
os.makedirs(SAVE_DIR, exist_ok=True)

# -- 1. Save model weights --
model_dir = os.path.join(SAVE_DIR, 'model')
os.makedirs(model_dir, exist_ok=True)
shutil.copy('best_dac_model.pt', os.path.join(model_dir, 'best_dac_model.pt'))

# -- 2. Save as HuggingFace format (for easy loading) --
hf_dir = os.path.join(model_dir, 'hf_format')
os.makedirs(hf_dir, exist_ok=True)

from transformers import AutoModelForSequenceClassification, AutoConfig

hf_config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2tag,
    label2id=tag2id
)
hf_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, config=hf_config
)

# Copy our trained weights into the HF model
hf_model.bert.load_state_dict(model.bert.state_dict())
hf_model.classifier.load_state_dict(model.classifier.state_dict())
hf_model.save_pretrained(hf_dir)
tokenizer.save_pretrained(hf_dir)
logger.info(f"HuggingFace model saved to {hf_dir}")

# -- 3. Save label map --
label_map = {'tag2id': tag2id, 'id2tag': {str(k): v for k, v in id2tag.items()}}
with open(os.path.join(model_dir, 'label_map.json'), 'w') as f:
    json.dump(label_map, f, indent=2)

# -- 4. Save classification report --
report = classification_report(
    test_labels, test_preds,
    target_names=TAGS, digits=4, zero_division=0
)
with open(os.path.join(SAVE_DIR, 'classification_report.txt'), 'w') as f:
    f.write(f"MediTOD DAC -- Test Set Classification Report\n")
    f.write(f"Best Epoch: {best_epoch}\n")
    f.write(f"Val F1: {best_val_f1:.4f}\n")
    f.write(f"Test F1: {test_f1:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.4f}\n\n")
    f.write(report)

# -- 5. Save training history --
with open(os.path.join(SAVE_DIR, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)

# -- 6. Copy plots --
for plot_file in ['training_curves.png', 'confusion_matrix.png']:
    if os.path.exists(plot_file):
        shutil.copy(plot_file, os.path.join(SAVE_DIR, plot_file))

# -- 7. Save hyperparameters --
hparams = {
    'model_name': MODEL_NAME,
    'max_seq_len': MAX_SEQ_LEN,
    'context_window': CONTEXT_WINDOW,
    'batch_size': BATCH_SIZE,
    'num_epochs': NUM_EPOCHS,
    'backbone_lr': BACKBONE_LR,
    'classifier_lr': CLASSIFIER_LR,
    'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO,
    'focal_gamma': FOCAL_GAMMA,
    'dropout': DROPOUT,
    'patience': PATIENCE,
    'freeze_layers': FREEZE_LAYERS,
    'best_epoch': best_epoch,
    'best_val_f1': best_val_f1,
    'test_f1': test_f1,
    'test_accuracy': test_acc,
}
with open(os.path.join(SAVE_DIR, 'hyperparameters.json'), 'w') as f:
    json.dump(hparams, f, indent=2)

# -- 8. Create zip archive --
zip_path = '/kaggle/working/meditod_dac_output'
shutil.make_archive(zip_path, 'zip', SAVE_DIR)
zip_size = os.path.getsize(f"{zip_path}.zip") / (1024 * 1024)

print(f"\n{'=' * 70}")
print(f"  ALL OUTPUTS SAVED")
print(f"{'=' * 70}")
print(f"  Model directory:       {model_dir}/")
print(f"  HuggingFace format:    {hf_dir}/")
print(f"  Classification report: {SAVE_DIR}/classification_report.txt")
print(f"  Training history:      {SAVE_DIR}/training_history.json")
print(f"  Hyperparameters:       {SAVE_DIR}/hyperparameters.json")
print(f"  Training curves:       {SAVE_DIR}/training_curves.png")
print(f"  Confusion matrix:      {SAVE_DIR}/confusion_matrix.png")
print(f"  Download ZIP:          {zip_path}.zip ({zip_size:.1f} MB)")
print(f"{'=' * 70}")